# Power Analysis — TXB_23
## Base de Dados Social | Events Page CTA Test

We calculate the minimum sample size required to detect a meaningful effect
for our two primary KPIs, using the pre/control group as baseline.

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import math
from io import StringIO

df = pd.read_csv('../data/TXB_23_landingpage_clean.csv', parse_dates=['arrival_time','exit_time'])
print(df.shape)

(1163, 16)


## Baseline Metrics

We use the pre/control group to establish baseline conversion rates for our two KPIs:
- **cta_click** (`kpi_y`) — binary, whether the visitor clicked the CTA button
- **enrolled** — binary, whether the visitor enrolled for an event (already created in `01_EDA.ipynb` as `kpi_x >= 50` AND `kpi_y == 1`)

In [ ]:
control = df[df['arm'] == 'pre'].copy()

# 'enrolled' column already exists in the clean CSV (created in 01_EDA.ipynb)
print(f"Control group size: {len(control)}")
print(f"CTA Click baseline rate:   {control['kpi_y'].mean():.4f}")
print(f"Enrollment baseline rate:  {control['enrolled'].mean():.4f}")

## Sample Size Calculation

Using the formula for two-sided tests with power=0.80 and α=0.05.
Target rates are set based on the minimum detectable effect we care about.

In [ ]:
# CTA Click (binary)
cta_mu_0 = control['kpi_y'].mean()
cta_mu_1 = 0.12
cta_sigma = math.sqrt(cta_mu_0 * (1 - cta_mu_0))
cta_effect_size = cta_mu_1 - cta_mu_0

# Enrollment (binary — now uses proportion formula)
enroll_mu_0 = control['enrolled'].mean()
enroll_mu_1 = 0.05   # target: double the enrollment rate
enroll_sigma = math.sqrt(enroll_mu_0 * (1 - enroll_mu_0))
enroll_effect_size = enroll_mu_1 - enroll_mu_0

# Power and significance
power = 0.80
alpha = 0.05
z_alpha_2 = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(power)

# Sample sizes (two-sample z-test formula for proportions)
cta_n = math.ceil(((z_alpha_2 + z_beta)**2 * 2 * cta_sigma**2) / cta_effect_size**2)
enroll_n = math.ceil(((z_alpha_2 + z_beta)**2 * 2 * enroll_sigma**2) / enroll_effect_size**2)

print(f"CTA Click    — baseline: {cta_mu_0:.4f}, target: {cta_mu_1}, required n per group: {cta_n}")
print(f"Enrollment   — baseline: {enroll_mu_0:.4f}, target: {enroll_mu_1}, required n per group: {enroll_n}")

## Power Analysis Summary

| KPI | Baseline | Target | Required n/group | Actual n (pre) | Actual n (treatment) | Powered? |
|-----|----------|--------|-----------------|----------------|----------------------|----------|
| CTA Click (`kpi_y`) | 0.0888 | 0.12 | 1,308 | 349 | 814 | No |
| Enrollment (`enrolled`) | 0.0258 | 0.05 | 673 | 349 | 814 | No |

Both KPIs are **underpowered** given our sample sizes. The control group (349) falls 
below the minimum required for both metrics. This means our test may not have 
enough statistical power to reliably detect the desired effect sizes. Results should 
be interpreted with caution — particularly for enrollment, where the baseline rate is 
very low (~2.6%) and small differences require large samples to detect.